# Debug Split Dataset

This notebook loads the split dataset the same way OpenPI does and checks for issues with video frame counts and timestamps.


In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import pandas as pd
import json
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
import subprocess
import numpy as np


## 1. Load the split dataset metadata


In [ ]:
# Path to the split dataset
split_dataset_root = Path("/Users/sherrychen/Documents/chef_dev/ChefResearch/sandi/datasets/chef_robotics/lettuce-sandwich-skills")

# Load info.json to check FPS and other metadata
with open(split_dataset_root / "meta/info.json") as f:
    info = json.load(f)

print("Dataset info:")
print(f"  FPS: {info['fps']}")
print(f"  Total episodes: {info['total_episodes']}")
print(f"  Total frames: {info['total_frames']}")
print(f"  Video keys: {info['video_keys']}")
print()


## 2. Check a few episodes for timestamp and frame count consistency


In [ ]:
def get_video_frame_count(video_path):
    """Get the actual frame count from a video file using ffprobe."""
    try:
        cmd = [
            "ffprobe",
            "-v", "error",
            "-select_streams", "v:0",
            "-count_packets",
            "-show_entries", "stream=nb_read_packets",
            "-of", "csv=p=0",
            str(video_path)
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return int(result.stdout.strip())
    except Exception as e:
        print(f"Error getting frame count for {video_path}: {e}")
        return None

def get_video_duration(video_path):
    """Get the duration of a video file using ffprobe."""
    try:
        cmd = [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "csv=p=0",
            str(video_path)
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return float(result.stdout.strip())
    except Exception as e:
        print(f"Error getting duration for {video_path}: {e}")
        return None

def get_video_fps(video_path):
    """Get the FPS of a video file using ffprobe."""
    try:
        cmd = [
            "ffprobe",
            "-v", "error",
            "-select_streams", "v:0",
            "-show_entries", "stream=r_frame_rate",
            "-of", "csv=p=0",
            str(video_path)
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        # Parse fraction like "30000/1001"
        nums = result.stdout.strip().split('/')
        return float(nums[0]) / float(nums[1]) if len(nums) == 2 else float(nums[0])
    except Exception as e:
        print(f"Error getting FPS for {video_path}: {e}")
        return None


In [ ]:
# Check first 5 episodes
num_episodes_to_check = 5
video_key = info['video_keys'][0]  # e.g., 'observation.images.cam_high'
fps = info['fps']

print(f"Checking first {num_episodes_to_check} episodes...")
print(f"Using video key: {video_key}")
print(f"Expected FPS: {fps}")
print("="*100)

issues_found = []

for ep_idx in range(num_episodes_to_check):
    chunk_idx = ep_idx // info['chunks_size']
    
    # Load parquet data
    parquet_path = split_dataset_root / f"data/chunk-{chunk_idx:03d}/episode_{ep_idx:06d}.parquet"
    if not parquet_path.exists():
        print(f"Episode {ep_idx}: Parquet file not found")
        continue
    
    df = pd.read_parquet(parquet_path)
    
    # Get video path
    video_path = split_dataset_root / f"videos/chunk-{chunk_idx:03d}/{video_key}/episode_{ep_idx:06d}.mp4"
    if not video_path.exists():
        print(f"Episode {ep_idx}: Video file not found")
        continue
    
    # Get actual video properties
    video_frame_count = get_video_frame_count(video_path)
    video_duration = get_video_duration(video_path)
    video_fps = get_video_fps(video_path)
    
    # Get parquet properties
    parquet_frame_count = len(df)
    parquet_timestamps = df['timestamp'].values
    parquet_first_ts = parquet_timestamps[0]
    parquet_last_ts = parquet_timestamps[-1]
    
    print(f"\nEpisode {ep_idx}:")
    print(f"  Parquet:")
    print(f"    Frame count: {parquet_frame_count}")
    print(f"    First timestamp: {parquet_first_ts:.6f}s")
    print(f"    Last timestamp: {parquet_last_ts:.6f}s")
    print(f"    Duration (last - first): {(parquet_last_ts - parquet_first_ts):.6f}s")
    print(f"  Video:")
    print(f"    Frame count: {video_frame_count}")
    print(f"    Duration: {video_duration:.6f}s")
    print(f"    FPS: {video_fps:.6f}")
    print(f"  Analysis:")
    print(f"    Frame count match: {parquet_frame_count == video_frame_count}")
    
    # Check what frame index the last timestamp would map to
    expected_last_frame_idx = round(parquet_last_ts * fps)
    print(f"    Last timestamp would map to frame index: {expected_last_frame_idx} (using round(ts * fps))")
    print(f"    Video has frames: 0 to {video_frame_count - 1}")
    
    if parquet_frame_count != video_frame_count:
        msg = f"Episode {ep_idx}: Frame count mismatch! Parquet={parquet_frame_count}, Video={video_frame_count}"
        print(f"    ⚠️  WARNING: {msg}")
        issues_found.append(msg)
    
    if expected_last_frame_idx >= video_frame_count:
        msg = f"Episode {ep_idx}: Last timestamp maps to invalid frame! round({parquet_last_ts:.6f} * {fps}) = {expected_last_frame_idx}, but video only has {video_frame_count} frames (0-{video_frame_count-1})"
        print(f"    ❌ ERROR: {msg}")
        issues_found.append(msg)
    
    print("="*100)

print(f"\n\nSummary: Found {len(issues_found)} issues")
for issue in issues_found:
    print(f"  - {issue}")


## 3. Load dataset using OpenPI's method


In [ ]:
# Load dataset metadata
repo_id = "chef_robotics/lettuce-sandwich-skills"
local_root = split_dataset_root.parent

print(f"Loading dataset with repo_id={repo_id}, root={local_root}")
dataset_meta = LeRobotDatasetMetadata(repo_id, root=local_root)

print(f"\nDataset metadata:")
print(f"  FPS: {dataset_meta.fps}")
print(f"  Total episodes: {dataset_meta.total_episodes}")
print(f"  Total frames: {dataset_meta.total_frames}")
print(f"  Video keys: {dataset_meta.video_keys}")


In [ ]:
# Create dataset (similar to OpenPI's create_torch_dataset)
action_horizon = 10  # typical value for OpenPI
action_sequence_keys = ["action"]  # typical for OpenPI

dataset = LeRobotDataset(
    repo_id,
    root=local_root,
    delta_timestamps={
        key: [t / dataset_meta.fps for t in range(action_horizon)] for key in action_sequence_keys
    },
)

print(f"Dataset loaded: {len(dataset)} samples")
print(f"Episodes: {len(dataset.episode_data_index['from'])} episodes")


## 4. Try to load samples at episode boundaries (most likely to fail)


In [ ]:
print("Testing samples at episode boundaries (last frames are most likely to have issues)...\n")
print("="*100)

failed_samples = []

for ep_idx in range(min(10, dataset_meta.total_episodes)):
    ep_start = dataset.episode_data_index['from'][ep_idx].item()
    ep_end = dataset.episode_data_index['to'][ep_idx].item()
    ep_length = ep_end - ep_start
    
    print(f"\nEpisode {ep_idx}: {ep_length} frames, dataset indices [{ep_start}, {ep_end})")
    
    # Try to load the last frame (most likely to fail)
    last_idx = ep_end - 1
    try:
        sample = dataset[last_idx]
        ts = sample['timestamp'].item() if hasattr(sample['timestamp'], 'item') else sample['timestamp']
        print(f"  ✓ Last frame (idx={last_idx}): timestamp={ts:.6f}s")
    except Exception as e:
        error_msg = str(e)
        print(f"  ❌ Last frame (idx={last_idx}): FAILED")
        print(f"     Error: {error_msg[:200]}")
        failed_samples.append((ep_idx, last_idx, error_msg))
        
        # Get the timestamp from the hf_dataset to see what it was trying to load
        try:
            hf_sample = dataset.hf_dataset[last_idx]
            problem_ts = hf_sample['timestamp']
            print(f"     Timestamp in parquet: {problem_ts:.6f}s")
            print(f"     Would map to frame index: {round(problem_ts * fps)}")
        except:
            pass

print("\n" + "="*100)
print(f"\nSummary: {len(failed_samples)} samples failed to load")
for ep_idx, idx, error in failed_samples:
    print(f"  - Episode {ep_idx}, sample {idx}: {error[:100]}")


## 5. Diagnosis

**How OpenPI/LeRobot loads videos:**

1. When you request a sample, LeRobot looks up the timestamp in the parquet file
2. It converts the timestamp to a frame index using: `frame_index = round(timestamp * fps)`
3. It tries to load that frame from the video using `decoder.get_frames_at(indices=[frame_index])`
4. If the frame index >= video frame count, it crashes with "Invalid frame index"

**The problem:**

When we split the dataset, we likely created a mismatch between:
- The timestamps in the parquet files 
- The actual number of frames in the split videos

This happens because:
- FFmpeg video splitting may not produce exact frame counts
- We're resetting timestamps to start from 0, but the video duration may be slightly different
- Rounding differences between `int(end_time * fps)` and `round(timestamp * fps)`
